# Tutorial 2 — Pre-trained Language Models và PhoBERT cho phân loại câu tiếng Việt

**Mục tiêu:** hiểu `pre-training → fine-tuning`; BERT/RoBERTa/PhoBERT; tải dataset sentiment tiếng Việt từ Kaggle; xây TF-IDF baseline; fine-tune `vinai/phobert-base-v2`; đánh giá Accuracy/Macro-F1; inference câu mới.

> Khuyến nghị Colab: `Runtime → Change runtime type → T4 GPU`.

### Dataset
**Synthetic Vietnamese Students' Feedback Corpus** trên Kaggle  
Dataset handle: `toreleon/synthetic-vietnamese-students-feedback-corpus`

Dataset có hơn 10,000 câu feedback, với sentiment positive/negative/neutral và topic. Trong tutorial này ta chỉ dùng **sentiment classification**.

## 1. Từ supervised learning đến Pre-trained Language Model

Cách truyền thống:

```text
Dataset nhỏ → TF-IDF / word representation → Classifier
```

Pre-trained LM:

```text
Rất nhiều text không nhãn
        ↓
    PRE-TRAIN
        ↓
Pre-trained Language Model
        ↓
Dataset có nhãn
        ↓
    FINE-TUNE
        ↓
Downstream task
```

Ý tưởng: **không học ngôn ngữ từ đầu cho từng task**.

## 2. BERT, RoBERTa và PhoBERT

**BERT** là Transformer encoder hai chiều. Một mục tiêu nổi tiếng là Masked Language Modeling:

> Tôi rất `<mask>` bộ phim này.

**RoBERTa** giữ kiến trúc encoder kiểu BERT nhưng tối ưu lại quá trình pre-training.

**PhoBERT** là pretrained language model đơn ngữ tiếng Việt theo hướng RoBERTa.

Theo model card chính thức, `vinai/phobert-base-v2`:

- khoảng **135M parameters**;
- maximum length **256**;
- pre-training trên Wikipedia + News và thêm dữ liệu OSCAR;
- tokenizer BPE;
- input nên được **word-segmented** trước khi đưa vào model.

Classification pipeline:

```text
Vietnamese sentence
   ↓
Word segmentation
   ↓
PhoBERT tokenizer
   ↓
PhoBERT encoder
   ↓
Classification head
   ↓
Positive / Neutral / Negative
```

## 3. Fine-tuning

Ta thêm classification head vào biểu diễn pretrained:

\[
p(y|x)=\mathrm{softmax}(Wh+b)
\]

và tối ưu cross-entropy:

\[
\mathcal{L}=-\sum_c y_c\log p_c
\]

Trong **full fine-tuning**, cả pretrained encoder và classification head đều được cập nhật.

Learning rate thường nhỏ, ví dụ `2e-5`, vì ta muốn **thích nghi** model đã học sẵn thay vì học lại từ đầu.

In [ ]:
!pip -q install -U transformers kagglehub underthesea scikit-learn

import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from underthesea import word_tokenize
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 4. Download dataset từ Kaggle

Ta dùng `kagglehub`. Với dataset công khai thường không cần upload `kaggle.json`. Nếu Kaggle yêu cầu xác thực, làm theo hướng dẫn API token của Kaggle.

In [ ]:
import kagglehub

DATASET_HANDLE = "toreleon/synthetic-vietnamese-students-feedback-corpus"
dataset_path = kagglehub.dataset_download(DATASET_HANDLE)

print("Dataset folder:", dataset_path)

all_files = [
    f for f in glob.glob(os.path.join(dataset_path, "**", "*"), recursive=True)
    if os.path.isfile(f)
]
for f in all_files:
    print("-", f)

## 5. Đọc và kiểm tra dữ liệu

Code dưới đây tự tìm CSV/XLSX/JSON và suy ra cột text/label.  
Trong bài tập thực tế, sinh viên **phải kiểm tra lại** cột mà code chọn.

In [ ]:
def load_first_table(folder):
    files = []
    for pat in ["**/*.csv","**/*.xlsx","**/*.xls","**/*.json"]:
        files += glob.glob(os.path.join(folder, pat), recursive=True)

    if not files:
        raise FileNotFoundError("Không tìm thấy CSV/XLSX/JSON.")

    file = files[0]
    print("Reading:", file)

    if file.lower().endswith(".csv"):
        return pd.read_csv(file, sep=None, engine="python")
    if file.lower().endswith((".xlsx",".xls")):
        return pd.read_excel(file)
    return pd.read_json(file)

df_raw = load_first_table(dataset_path)
print("Shape:", df_raw.shape)
display(df_raw.head())
print("Columns:", list(df_raw.columns))

In [ ]:
def infer_text_column(df):
    priority = ["feedback","text","sentence","comment","review","content","utterance"]
    for key in priority:
        for c in df.columns:
            if key in str(c).lower():
                return c

    object_cols = [
        c for c in df.columns
        if pd.api.types.is_object_dtype(df[c])
        or pd.api.types.is_string_dtype(df[c])
    ]
    if not object_cols:
        raise ValueError("Không tìm thấy cột text.")

    return max(
        object_cols,
        key=lambda c: df[c].astype(str).str.len().mean()
    )

def infer_label_column(df, text_col):
    priority = ["sentiment","label","polarity","class"]
    for key in priority:
        for c in df.columns:
            if c != text_col and key in str(c).lower():
                return c

    candidates = []
    for c in df.columns:
        if c == text_col:
            continue
        n = df[c].nunique(dropna=True)
        if 2 <= n <= 10:
            candidates.append((c,n))

    if not candidates:
        raise ValueError("Không suy ra được label column.")
    return sorted(candidates, key=lambda x:x[1])[0][0]

TEXT_COL = infer_text_column(df_raw)
LABEL_COL = infer_label_column(df_raw, TEXT_COL)

print("TEXT_COL :", TEXT_COL)
print("LABEL_COL:", LABEL_COL)

df = df_raw[[TEXT_COL, LABEL_COL]].copy()
df.columns = ["text","label"]
df = df.dropna()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()
df = df[df["text"].str.len() > 0].reset_index(drop=True)

display(df.head())
display(df["label"].value_counts())

## 6. Chế độ lab nhanh

Để fine-tuning nhanh trong một buổi học, mặc định dùng tối đa khoảng 3000 mẫu.  
Đổi `MAX_SAMPLES=None` để dùng toàn bộ dataset.

In [ ]:
MAX_SAMPLES = 3000

if MAX_SAMPLES is not None and len(df) > MAX_SAMPLES:
    parts = []
    per_class = max(1, MAX_SAMPLES // df["label"].nunique())
    for label, group in df.groupby("label"):
        parts.append(
            group.sample(
                n=min(per_class, len(group)),
                random_state=SEED
            )
        )
    df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Dataset used:", len(df))
display(df["label"].value_counts())

## 7. Encode label và chia train/test
Dùng stratified split để giữ tỷ lệ lớp.

In [ ]:
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

id2label = {int(i):str(v) for i,v in enumerate(label_encoder.classes_)}
label2id = {v:k for k,v in id2label.items()}
print(id2label)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label_id"]
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

# 8. Baseline: TF-IDF + Logistic Regression

Một model phức tạp chỉ có ý nghĩa khi so với baseline hợp lý.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=30_000,
    min_df=2
)
X_train = tfidf.fit_transform(train_df["text"])
X_test = tfidf.transform(test_df["text"])

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train, train_df["label_id"])
lr_pred = lr.predict(X_test)

baseline_acc = accuracy_score(test_df["label_id"], lr_pred)
baseline_f1 = f1_score(test_df["label_id"], lr_pred, average="macro")

print(f"TF-IDF Accuracy: {baseline_acc:.4f}")
print(f"TF-IDF Macro-F1: {baseline_f1:.4f}")
print(classification_report(
    test_df["label_id"], lr_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
))

# 9. Word segmentation cho PhoBERT

PhoBERT được pretrain trên tiếng Việt đã word-segmented. Ví dụ:

```text
sinh viên học máy
→
sinh_viên học_máy
```

Model card chính thức khuyến nghị segmentation tương thích với VnCoreNLP.  
Tutorial dùng `underthesea` để giảm độ phức tạp cài đặt. Khi làm nghiên cứu, nên thử lại bằng **VnCoreNLP/RDRSegmenter**.

In [ ]:
def segment_vi(text):
    return word_tokenize(str(text), format="text")

example = "Giảng viên dạy rất dễ hiểu và nhiệt tình."
print("Raw      :", example)
print("Segmented:", segment_vi(example))

In [ ]:
train_texts = [
    segment_vi(x)
    for x in tqdm(train_df["text"].tolist(), desc="Segment train")
]
test_texts = [
    segment_vi(x)
    for x in tqdm(test_df["text"].tolist(), desc="Segment test")
]

train_labels = train_df["label_id"].astype(int).tolist()
test_labels = test_df["label_id"].astype(int).tolist()

print(train_texts[:3])

# 10. PhoBERT tokenizer

Dùng `vinai/phobert-base-v2`. Với feedback ngắn, `MAX_LENGTH=128` đủ cho tutorial và giúp tiết kiệm GPU.

In [ ]:
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

sample = tokenizer(
    train_texts[0],
    truncation=True,
    max_length=MAX_LENGTH
)

print(train_texts[0])
print(sample["input_ids"][:30])
print(tokenizer.convert_ids_to_tokens(sample["input_ids"][:30]))

## 11. PyTorch Dataset

Mỗi sample gồm `input_ids`, `attention_mask`, `labels`.

In [ ]:
class PhoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k:v[idx] for k,v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = PhoBERTDataset(
    train_texts, train_labels, tokenizer, MAX_LENGTH
)
test_dataset = PhoBERTDataset(
    test_texts, test_labels, tokenizer, MAX_LENGTH
)

BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

batch = next(iter(train_loader))
for k,v in batch.items():
    print(k, v.shape)

# 12. Pretrained encoder + classification head

Classification head mới được khởi tạo; PhoBERT encoder lấy trọng số pretrained.

In [ ]:
num_labels = len(id2label)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}")

# 13. Full fine-tuning

Thiết lập:

- AdamW
- learning rate `2e-5`
- 2 epochs
- linear scheduler + 10% warmup

In [ ]:
EPOCHS = 2
LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

num_training_steps = EPOCHS * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

print("Training steps:", num_training_steps)

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            batch = {k:v.to(device) for k,v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()

            preds = outputs.logits.argmax(dim=-1)
            all_preds += preds.cpu().tolist()
            all_labels += batch["labels"].cpu().tolist()

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss/len(loader), acc, f1, all_labels, all_preds


history = []

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in progress:
        batch = {k:v.to(device) for k,v in batch.items()}
        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_f1, _, _ = evaluate_model(model, test_loader)

    history.append({
        "epoch":epoch+1,
        "train_loss":train_loss,
        "val_loss":val_loss,
        "accuracy":val_acc,
        "macro_f1":val_f1
    })

    print(
        f"Epoch {epoch+1}: train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | acc={val_acc:.4f} | "
        f"macro_f1={val_f1:.4f}"
    )

## 14. Đánh giá PhoBERT
Với multi-class, Macro-F1 giúp mỗi lớp có trọng số như nhau.

In [ ]:
test_loss, phobert_acc, phobert_f1, y_true, y_pred = evaluate_model(
    model, test_loader
)

print(f"PhoBERT Accuracy: {phobert_acc:.4f}")
print(f"PhoBERT Macro-F1: {phobert_f1:.4f}")

print(classification_report(
    y_true, y_pred,
    target_names=[id2label[i] for i in range(num_labels)]
))

In [ ]:
print("=== Comparison ===")
print(f"TF-IDF  | Accuracy={baseline_acc:.4f} | Macro-F1={baseline_f1:.4f}")
print(f"PhoBERT | Accuracy={phobert_acc:.4f} | Macro-F1={phobert_f1:.4f}")

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=[id2label[i] for i in range(num_labels)],
    xticks_rotation=45
)
plt.title("PhoBERT — Confusion Matrix")
plt.show()

history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(7,4))
plt.plot(history_df["epoch"], history_df["train_loss"],
         marker="o", label="Train")
plt.plot(history_df["epoch"], history_df["val_loss"],
         marker="o", label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.show()

# 15. Inference với câu mới

In [ ]:
def predict_text(text):
    model.eval()
    segmented = segment_vi(text)

    encoded = tokenizer(
        segmented,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH
    )
    encoded = {k:v.to(device) for k,v in encoded.items()}

    with torch.no_grad():
        probs = torch.softmax(model(**encoded).logits, dim=-1)[0]

    pred_id = int(probs.argmax())

    print("Raw      :", text)
    print("Segmented:", segmented)
    print("Prediction:", id2label[pred_id])

    display(pd.DataFrame({
        "label":[id2label[i] for i in range(num_labels)],
        "probability":probs.cpu().numpy()
    }).sort_values("probability", ascending=False))

predict_text("Giảng viên dạy rất dễ hiểu và luôn hỗ trợ sinh viên.")
predict_text("Môn học quá chán và nội dung rất khó hiểu.")

# 16. Feature extraction, full fine-tuning và PEFT

**Feature extraction:** freeze PhoBERT, chỉ train classifier.

**Full fine-tuning:** cập nhật toàn bộ model — cách dùng trong notebook.

**PEFT:** LoRA/adapters, chỉ cập nhật một lượng nhỏ tham số; phù hợp bài mở rộng.

# 17. Bài tập

1. So sánh TF-IDF unigram với unigram+bigram.
2. Chạy với 500 / 1000 / 3000 / toàn bộ mẫu và vẽ learning curve.
3. Freeze encoder rồi chỉ train classification head.
4. So sánh learning rate `1e-5`, `2e-5`, `5e-5`.
5. Error analysis 10 câu dự đoán sai: phủ định, sarcasm, slang, typo, mixed sentiment.
6. Thay PhoBERT bằng một Vietnamese pretrained encoder khác, nhưng giữ nguyên split/metric.
7. Nâng cao: fine-tune bằng LoRA và so số trainable parameters.

# 18. Tổng kết

Hai hệ thống:

```text
TF-IDF → Logistic Regression
```

và:

```text
Word segmentation
 → PhoBERT tokenizer
 → PhoBERT-base-v2
 → Classification head
```

Các điểm cần nhớ:

- pre-training học biểu diễn ngôn ngữ trên corpus lớn;
- fine-tuning thích nghi model cho downstream task;
- preprocessing cần tương thích với pre-training;
- luôn có baseline đơn giản;
- với multi-class, nên báo cáo cả Accuracy và Macro-F1;
- error analysis rất quan trọng.

### Tài liệu đọc thêm
- Nguyen & Nguyen, **PhoBERT: Pre-trained language models for Vietnamese**, Findings of EMNLP 2020.
- Hugging Face model card: `vinai/phobert-base-v2`.
- Devlin et al., **BERT**.
- Liu et al., **RoBERTa**.